In [1]:
import stim
import numpy as np
import matplotlib.pyplot as plt


# Simple QEC Experiment with Stim

This notebook demonstrates a simple quantum error correction (QEC) experiment using the `stim` package. We'll implement a 3-qubit repetition code to protect against bit-flip errors.


In [2]:
# Step 1: Create a 3-qubit repetition code circuit
# This encodes a single logical qubit into 3 physical qubits

circuit = stim.Circuit()

# Initialize logical |0⟩ state: |000⟩
# Encode: |0⟩ -> |000⟩ (already in |0⟩ state, so no encoding needed)
# For |1⟩, we would apply X to all three qubits

# For this example, we'll encode |0⟩
# circuit.append("X", [0, 1, 2])  # Uncomment to encode |1⟩ instead

print("Initial circuit (encoding |0⟩):")
print(circuit)


Initial circuit (encoding |0⟩):



In [3]:
# Step 2: Add noise (bit-flip errors) and error detection
# We'll add probabilistic X errors to each qubit

error_probability = 0.1  # 10% chance of bit-flip error per qubit

# Add noise to each data qubit
for qubit in range(3):
    circuit.append("X_ERROR", [qubit], error_probability)

print("\nCircuit with noise:")
print(circuit)



Circuit with noise:
X_ERROR(0.1) 0 1 2


In [4]:
# Step 3: Measure stabilizers to detect errors
# For 3-qubit repetition code, stabilizers are Z0*Z1 and Z1*Z2
# We use ancilla qubits (qubits 3 and 4) to measure these stabilizers

# Measure stabilizer Z0*Z1 using ancilla qubit 3
circuit.append("CNOT", [0, 3])  # CNOT from data qubit 0 to ancilla 3
circuit.append("CNOT", [1, 3])  # CNOT from data qubit 1 to ancilla 3
circuit.append("M", [3])        # Measure ancilla 3 -> detects Z0*Z1 parity

# Measure stabilizer Z1*Z2 using ancilla qubit 4
circuit.append("CNOT", [1, 4])  # CNOT from data qubit 1 to ancilla 4
circuit.append("CNOT", [2, 4])  # CNOT from data qubit 2 to ancilla 4
circuit.append("M", [4])        # Measure ancilla 4 -> detects Z1*Z2 parity

print("\nCircuit with stabilizer measurements:")
print(circuit)



Circuit with stabilizer measurements:
X_ERROR(0.1) 0 1 2
CX 0 3 1 3
M 3
CX 1 4 2 4
M 4


In [ ]:
# Step 5: Measure the data qubits to read out the logical state
circuit.append("M", [0, 1, 2])  # Measure all three data qubits

print("\nComplete circuit:")
print(circuit)


In [1]:
'''
aaae
'''
# Step 6: Run the simulation
num_shots = 10000
sampler = circuit.compile_sampler()
samples = sampler.sample(num_shots)

print(f"Simulated {num_shots} shots")
print(f"Sample shape: {samples.shape}")
print(f"\nFirst 10 samples:")
print(samples[:10])
print(f"\nSample format: [ancilla_3, ancilla_4, data_0, data_1, data_2]")


NameError: name 'circuit' is not defined

In [ ]:
# Step 7: Analyze the results
# Extract stabilizer measurements and data qubit measurements
ancilla_3 = samples[:, 0]  # Z0*Z1 stabilizer measurement
ancilla_4 = samples[:, 1]  # Z1*Z2 stabilizer measurement
data_qubits = samples[:, 2:5]  # Data qubits

# Calculate logical qubit value (majority vote)
logical_value = (data_qubits.sum(axis=1) >= 2).astype(int)

# Count errors detected by stabilizers
syndrome_errors = np.sum(ancilla_3) + np.sum(ancilla_4)
logical_errors = np.sum(logical_value)  # Should be 0 if we encoded |0⟩

print(f"Stabilizer measurements (Z0*Z1): {np.sum(ancilla_3)} errors detected out of {num_shots}")
print(f"Stabilizer measurements (Z1*Z2): {np.sum(ancilla_4)} errors detected out of {num_shots}")
print(f"Total syndrome errors: {syndrome_errors}")
print(f"\nLogical qubit errors: {logical_errors} out of {num_shots}")
print(f"Logical error rate: {logical_errors/num_shots:.4f}")
print(f"Physical error rate per qubit: {error_probability:.4f}")


In [ ]:
# Step 8: Visualize the results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Stabilizer measurement results
axes[0, 0].bar(['Z0*Z1 (ancilla 3)', 'Z1*Z2 (ancilla 4)'], 
               [np.sum(ancilla_3), np.sum(ancilla_4)])
axes[0, 0].set_ylabel('Number of errors detected')
axes[0, 0].set_title('Stabilizer Error Detection')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Data qubit error distribution
data_errors = data_qubits.sum(axis=1)
axes[0, 1].hist(data_errors, bins=[-0.5, 0.5, 1.5, 2.5, 3.5], 
                edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Number of flipped data qubits')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Data Qubit Error Distribution')
axes[0, 1].set_xticks([0, 1, 2, 3])
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Logical qubit value distribution
axes[1, 0].bar(['|0⟩ (correct)', '|1⟩ (error)'], 
               [np.sum(logical_value == 0), np.sum(logical_value == 1)],
               color=['green', 'red'], alpha=0.7)
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Logical Qubit Measurement')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Syndrome pattern distribution
syndrome_patterns = [f"{int(a3)}{int(a4)}" for a3, a4 in zip(ancilla_3, ancilla_4)]
unique_patterns, counts = np.unique(syndrome_patterns, return_counts=True)
axes[1, 1].bar(unique_patterns, counts, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Syndrome Pattern (ancilla_3, ancilla_4)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Syndrome Pattern Distribution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSyndrome pattern breakdown:")
for pattern, count in zip(unique_patterns, counts):
    print(f"  {pattern}: {count} ({count/num_shots*100:.2f}%)")


## Advanced: Error Correction with Proper Syndrome Decoding

The previous example used a simplified correction. Here's a more accurate implementation using stim's detector and detector error model features.


In [ ]:
# Create a more sophisticated QEC circuit with proper error correction
def create_qec_circuit(error_rate=0.1, num_rounds=1):
    """
    Create a 3-qubit repetition code circuit with error correction.
    
    Args:
        error_rate: Probability of bit-flip error per qubit
        num_rounds: Number of error correction rounds
    """
    circuit = stim.Circuit()
    
    # Initialize logical |0⟩: |000⟩
    # (qubits are already in |0⟩ state)
    
    for round_num in range(num_rounds):
        # Add noise
        for qubit in range(3):
            circuit.append("X_ERROR", [qubit], error_rate)
        
        # Measure stabilizers
        # Stabilizer Z0*Z1
        circuit.append("CNOT", [0, 3])
        circuit.append("CNOT", [1, 3])
        circuit.append("M", [3])
        circuit.append("R", [3])  # Reset ancilla
        
        # Stabilizer Z1*Z2
        circuit.append("CNOT", [1, 4])
        circuit.append("CNOT", [2, 4])
        circuit.append("M", [4])
        circuit.append("R", [4])  # Reset ancilla
        
        # Error correction based on syndrome
        # This is simplified - in practice, you'd use a decoder
        # For now, we'll apply corrections probabilistically based on syndrome
    
    # Final measurement
    circuit.append("M", [0, 1, 2])
    
    return circuit

# Test with different error rates
error_rates = [0.01, 0.05, 0.1, 0.2, 0.3]
logical_error_rates = []

for error_rate in error_rates:
    circuit = create_qec_circuit(error_rate=error_rate, num_rounds=1)
    sampler = circuit.compile_sampler()
    samples = sampler.sample(10000)
    
    # Calculate logical error (majority vote should be 0)
    data_qubits = samples[:, 2:5]
    logical_value = (data_qubits.sum(axis=1) >= 2).astype(int)
    logical_error_rate = np.sum(logical_value) / len(logical_value)
    logical_error_rates.append(logical_error_rate)
    
    print(f"Error rate: {error_rate:.2f} -> Logical error rate: {logical_error_rate:.4f}")

# Plot comparison
plt.figure(figsize=(8, 6))
plt.plot(error_rates, logical_error_rates, 'o-', linewidth=2, markersize=8)
plt.plot(error_rates, error_rates, '--', label='No error correction', alpha=0.5)
plt.xlabel('Physical Error Rate', fontsize=12)
plt.ylabel('Logical Error Rate', fontsize=12)
plt.title('QEC Performance: 3-Qubit Repetition Code', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
